# AI Telecaller — Local Proof-of-Concept (Colab, T4 GPU)

**Scope of this notebook:** local pipeline only — STT → RAG → LLM → TTS — tested with typed
text or recorded/uploaded audio. **No telephony** (no PBX, no SIP trunk, no Twilio, no ngrok).
That is a later phase.

Pipeline this notebook builds, tested stage-by-stage:

1. **STT** — faster-whisper transcribes uploaded or mic-recorded audio.
2. **Conversation LLM** — a 4-bit quantized 7-8B instruct model (Qwen2.5-7B-Instruct by
   default), with an editable programme pitch + consent disclosure system prompt.
3. **RAG knowledge base** — Chroma (in-memory) + `all-MiniLM-L6-v2` embeddings. The LLM is
   **only** allowed to answer fee/date/certification questions using retrieved context —
   it must never invent these facts.
4. **TTS** — Piper, using real, verified voice names (`en_US-lessac-medium`,
   `en_GB-alan-medium`).
5. **Chain** — one function: (audio or text) → STT → RAG → LLM → TTS, with per-stage
   latency printed on every run.
6. **Test harness** — a fixed set of caller questions/objections run through the full
   chain automatically, including a "remove my number" case that must trigger a
   suppression signal instead of a normal reply.

Run cells top to bottom. Each component is tested immediately after it's built —
don't skip the verification cells.

**Colab setup:** Runtime → Change runtime type → **T4 GPU**.


## 0. Install dependencies

One-time setup. This cell **force-restarts the runtime itself** at the end (via
`os.kill`), because Colab preloads its own `numpy` into the kernel process before this
cell ever runs -- so even a fully correct `pip install` can't fix an already-imported
old `numpy` sitting in memory; only a fresh kernel process picks up the newly-installed
version. You will see a "Your session crashed for an unknown reason" notice right after
this cell finishes -- **that's expected, not an actual failure**, it's this restart.
Once Colab reconnects, don't re-run this cell -- everything it installed is already on
disk. Just continue to Section 1.

In [ ]:
# System package needed to decode the webm/ogg audio that the in-notebook mic recorder
# produces, and to let pydub do format conversion.
!apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null

!pip -q install "numpy==2.4.6" \
                "faster-whisper==1.2.1" \
                "transformers==5.16.1" "accelerate==1.14.0" "bitsandbytes==0.50.2" \
                "sentence-transformers==6.0.1" "chromadb==1.5.9" \
                "piper-tts==1.7.0" \
                "ipywebrtc==0.6.0" "pydub==0.25.1" "soundfile==0.12.1"
# Versions pinned exactly (checked against PyPI's current releases at the time
# of writing) so this notebook keeps working the same way months from now.
#
# numpy is pinned EXPLICITLY (not just left to whatever transformers/chromadb/
# etc. pull in transitively). This matters because pip only upgrades a
# dependency if the version already on disk doesn't satisfy the requirement --
# if a prior/different install already forced numpy down to e.g. 1.26.x, a
# loose "numpy>=1.17" from transformers is satisfied by 1.26.x and pip will
# NOT put numpy back up to 2.x on its own. Pinning it exactly here forces pip
# to always reconcile it to the version this notebook actually needs.
#
# The whole ML stack (transformers/accelerate/bitsandbytes/sentence-transformers/
# chromadb) is pinned to versions that all require numpy>=2 and huggingface-hub
# >=1.x -- matching what Colab's base image already has installed for gradio,
# jax, opencv, diffusers, etc. Older pins here (e.g. chromadb<0.6 wants
# numpy<2, transformers<5 wants huggingface-hub<1) force pip to downgrade
# numpy/huggingface-hub, which then breaks every other numpy>=2-requiring
# package already in the Colab image with "incompatible" resolver warnings.
# Separately, transformers>=4.47's tokenizers>=0.21 dependency ships "abi3"
# wheels (one wheel works on every Python 3.9+ release) instead of
# per-Python-version wheels, so this also avoids the from-source Rust build
# that older tokenizers pins fail with once Colab's Python moves past what
# that old release shipped wheels for.
#
# transformers v5 uses only the BitsAndBytesConfig quantization API (which is
# what this notebook already uses below), so no other code needed to change.
#
# IMPORTANT: if you've run an OLDER version of this install cell before in
# this same runtime, numpy may already be stuck at an old version on disk.
# Runtime -> Restart session BEFORE running this cell, and again AFTER it
# finishes, so the newly-installed numpy is actually loaded fresh.

print("Install step finished.")
print("Restarting the runtime now so the freshly installed numpy/torch/transformers")
print("load cleanly in a fresh process (Colab preloads its own numpy into this kernel")
print("before this cell ever runs, so a correct pip install alone can't replace an")
print("already-imported old numpy in memory). You'll see a 'session crashed for an")
print("unknown reason' notice -- that's expected, it's this restart, not a failure.")
print("After it reconnects, do NOT re-run this cell -- continue from Section 1.")

import os
os.kill(os.getpid(), 9)


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.61.2 requires numpy<2.3,>=1.24, but you have numpy 2.4.6 which is incompatible.


In [1]:
# Sanity check: import everything we'll rely on, and confirm a GPU is visible.
import torch, transformers, faster_whisper, sentence_transformers, chromadb
import piper as piper_tts_pkg

print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected. Runtime -> Change runtime type -> T4 GPU, then re-run.")

print("transformers:", transformers.__version__)
print("faster_whisper OK, sentence_transformers OK, chromadb OK, piper OK")


torch: 2.11.0+cu128 | CUDA available: True
GPU: Tesla T4
transformers: 5.16.1
faster_whisper OK, sentence_transformers OK, chromadb OK, piper OK


## 1. Speech-to-Text — faster-whisper

Loads a `faster-whisper` model and verifies it on a known public audio sample before we
trust it on anything else. Also provides a mic-recording helper for Colab so you don't
need a pre-made audio file every time.

In [2]:
import time
from faster_whisper import WhisperModel

# "small" is a good speed/accuracy balance on a free T4; bump to "medium" if you have
# quota to spare and want better accuracy.
WHISPER_MODEL_SIZE = "small"

whisper_model = WhisperModel(WHISPER_MODEL_SIZE, device="cuda" if torch.cuda.is_available() else "cpu",
                              compute_type="float16" if torch.cuda.is_available() else "int8")

def transcribe_audio(audio_path: str):
    '''Returns (text, elapsed_seconds).'''
    t0 = time.time()
    segments, info = whisper_model.transcribe(audio_path, beam_size=5)
    text = " ".join(seg.text.strip() for seg in segments).strip()
    elapsed = time.time() - t0
    return text, elapsed

print(f"Loaded faster-whisper '{WHISPER_MODEL_SIZE}'.")


Loaded faster-whisper 'small'.


In [3]:
# --- Verify STT actually works, on a known sample, before building on top of it ---
import urllib.request, os

TEST_AUDIO_PATH = "/content/jfk_sample.flac"
TEST_AUDIO_URL = "https://github.com/openai/whisper/raw/main/tests/jfk.flac"

if not os.path.exists(TEST_AUDIO_PATH):
    urllib.request.urlretrieve(TEST_AUDIO_URL, TEST_AUDIO_PATH)

size = os.path.getsize(TEST_AUDIO_PATH)
assert size > 1000, f"Downloaded test audio looks empty/broken ({size} bytes)"
print(f"Test audio downloaded OK: {size} bytes")

text, elapsed = transcribe_audio(TEST_AUDIO_PATH)
print(f"Transcribed in {elapsed:.2f}s:\n  {text!r}")

assert len(text) > 10, "STT returned suspiciously little text — investigate before continuing."
assert "country" in text.lower() or "ask" in text.lower(), \
    "Transcript doesn't look like the expected JFK sample — investigate before continuing."
print("STT stage VERIFIED.")


Test audio downloaded OK: 1152693 bytes
Transcribed in 2.32s:
  'And so my fellow Americans, ask not what your country can do for you, ask what you can do for your country.'
STT stage VERIFIED.


### 1a. Mic recording in Colab (optional — use this or upload a file)

`ipywebrtc`'s `AudioRecorder` records straight from your browser mic into the notebook.
If it doesn't render (some browsers/Colab versions are flaky with it), just upload a
`.wav`/`.mp3`/`.m4a` file instead with `from google.colab import files; files.upload()`
and pass that path to `transcribe_audio()` directly — the rest of the pipeline doesn't
care how the audio got onto disk.

In [4]:
from google.colab import output
output.enable_custom_widget_manager()

from ipywebrtc import AudioRecorder, CameraStream
from pydub import AudioSegment
import ipywidgets as widgets

def make_mic_recorder():
    '''Displays a mic recorder widget. Click the record button, speak, click again to stop.'''
    camera = CameraStream(constraints={'audio': True, 'video': False})
    recorder = AudioRecorder(stream=camera)
    display(recorder)
    return recorder

def save_recording_as_wav(recorder, out_path="/content/mic_recording.wav"):
    '''Call this AFTER you've stopped the recording. Converts the recorder's webm blob to wav.'''
    raw_path = "/content/_mic_raw.webm"
    with open(raw_path, "wb") as f:
        f.write(recorder.audio.value)
    audio = AudioSegment.from_file(raw_path)
    audio.export(out_path, format="wav")
    size = os.path.getsize(out_path)
    assert size > 100, f"Converted recording looks empty ({size} bytes) — did you actually record anything?"
    print(f"Saved recording: {out_path} ({size} bytes, {len(audio)/1000:.1f}s)")
    return out_path

# Example usage (uncomment to actually record in this cell):
# mic = make_mic_recorder()


In [5]:
# After recording above and clicking stop, run this in a NEW cell to save + transcribe it:
# wav_path = save_recording_as_wav(mic)
# text, elapsed = transcribe_audio(wav_path)
# print(text, elapsed)
print("Mic helper ready. See the commented example above for usage once you've recorded something.")


Mic helper ready. See the commented example above for usage once you've recorded something.


## 2. Conversation LLM — quantized 7-8B instruct model

### Edit these variables with the real programme details

Everything about the pitch — name, fee, dates, curriculum, certification — lives here so
you can update it without touching any pipeline code below.

In [6]:
# ============================================================
# EDIT ME: real programme details go here
# ============================================================
PROGRAMME_NAME = "Full-Stack AI Engineering Bootcamp"          # <-- edit
PROGRAMME_FEE = "INR 45,000 (plus applicable taxes)"           # <-- edit
PROGRAMME_DATES = "Batch starts 6 October 2026, runs for 8 weeks, weekday evenings 7-9 PM IST"  # <-- edit
PROGRAMME_DURATION = "8 weeks, 3 live sessions per week"        # <-- edit
PROGRAMME_CURRICULUM = [                                        # <-- edit
    "Python & ML fundamentals",
    "LLM application development (RAG, agents, fine-tuning basics)",
    "Deploying AI systems to production",
    "Capstone project with mentor review",
]
CERTIFICATION_NAME = "Certificate of Completion, co-signed by RagaTech Source"  # <-- edit
COMPANY_NAME = "RagaTech Source"                                # <-- edit
AGENT_DISPLAY_NAME = "Riya"                                     # <-- edit: the AI caller's on-call persona name

CONSENT_DISCLOSURE = (
    f"Hi, this is {AGENT_DISPLAY_NAME}, an AI assistant calling on behalf of {COMPANY_NAME}. "
    "This call is being conducted by an AI system and is recorded for quality and training "
    "purposes. Is it okay if I take a couple of minutes to tell you about a training "
    "programme we're running?"
)

print("Programme variables set:")
print(f"  {PROGRAMME_NAME} | {PROGRAMME_FEE} | {PROGRAMME_DATES}")


Programme variables set:
  Full-Stack AI Engineering Bootcamp | INR 45,000 (plus applicable taxes) | Batch starts 6 October 2026, runs for 8 weeks, weekday evenings 7-9 PM IST


In [ ]:
# ============================================================
# Model choice — 4-bit quantized 7-8B instruct model
# ============================================================
# Qwen2.5-7B-Instruct is ungated on the Hub (no HF token / license click-through needed),
# which makes it the simpler default for a fresh Colab session. Swap in
# "meta-llama/Llama-3.1-8B-Instruct" if you have accepted its license and set an HF token
# (huggingface_hub.login()) — no other code below needs to change.
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

t0 = time.time()
llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
print(f"Loaded {LLM_MODEL_ID} in 4-bit in {time.time() - t0:.1f}s")


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# --- Verify the LLM actually loads and responds before building on top of it ---
def llm_raw_generate(messages, max_new_tokens=200, temperature=0.4):
    '''messages: list of {"role": ..., "content": ...}. Returns (text, elapsed_seconds).'''
    prompt = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_model.device)
    t0 = time.time()
    with torch.no_grad():
        output_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            top_p=0.9,
            pad_token_id=llm_tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    text = llm_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    elapsed = time.time() - t0
    return text, elapsed

test_reply, test_elapsed = llm_raw_generate([
    {"role": "user", "content": "Say 'hello, I am working' and nothing else."}
], max_new_tokens=30)
print(f"LLM responded in {test_elapsed:.2f}s:\n  {test_reply!r}")
assert len(test_reply) > 0, "LLM returned an empty response — investigate before continuing."
print("LLM stage VERIFIED.")


## 3. Knowledge base (RAG) — Chroma + sentence-transformers

The LLM must **never invent** fee, date, or certification details. Those facts live here
as documents, get embedded with `all-MiniLM-L6-v2`, and are retrieved before every LLM
call so the model answers from grounded context instead of guessing.

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Facts are built from the editable variables above, so editing Section 2 automatically
# updates what the LLM is grounded on.
KNOWLEDGE_BASE = [
    {"id": "fee",           "text": f"The fee for the {PROGRAMME_NAME} is {PROGRAMME_FEE}."},
    {"id": "dates",         "text": f"The {PROGRAMME_NAME} schedule: {PROGRAMME_DATES}."},
    {"id": "duration",      "text": f"The {PROGRAMME_NAME} duration and format: {PROGRAMME_DURATION}."},
    {"id": "curriculum",    "text": f"The {PROGRAMME_NAME} curriculum covers: " + "; ".join(PROGRAMME_CURRICULUM) + "."},
    {"id": "certification", "text": f"On completing the {PROGRAMME_NAME}, participants receive: {CERTIFICATION_NAME}."},
]

chroma_client = chromadb.EphemeralClient()  # in-memory, no persistence, no Docker
# Drop and recreate so re-running this cell after editing KNOWLEDGE_BASE doesn't duplicate docs.
try:
    chroma_client.delete_collection("programme_kb")
except Exception:
    pass
kb_collection = chroma_client.create_collection("programme_kb")

kb_embeddings = embedder.encode([d["text"] for d in KNOWLEDGE_BASE]).tolist()
kb_collection.add(
    ids=[d["id"] for d in KNOWLEDGE_BASE],
    documents=[d["text"] for d in KNOWLEDGE_BASE],
    embeddings=kb_embeddings,
)

def retrieve_context(query: str, k: int = 2):
    '''Returns (context_text, [(doc, distance), ...]).'''
    q_emb = embedder.encode([query]).tolist()
    results = kb_collection.query(query_embeddings=q_emb, n_results=k)
    docs = results["documents"][0]
    dists = results["distances"][0]
    context_text = "\n".join(docs)
    return context_text, list(zip(docs, dists))

print(f"Knowledge base loaded: {len(KNOWLEDGE_BASE)} facts.")


In [ ]:
# --- Verify retrieval actually finds the right fact for each type of question ---
verify_queries = {
    "What is the fee for this programme?": PROGRAMME_FEE,
    "When does the programme start?": PROGRAMME_DATES.split(",")[0],
    "Is there a certificate at the end?": CERTIFICATION_NAME,
}

for query, expected_fragment in verify_queries.items():
    context, hits = retrieve_context(query, k=1)
    top_doc, top_dist = hits[0]
    print(f"Q: {query}\n  top hit (distance={top_dist:.3f}): {top_doc}")
    assert expected_fragment.split()[0] in top_doc or expected_fragment[:15].lower() in top_doc.lower(), \
        f"Retrieval didn't surface the expected fact for: {query!r}"

print("RAG retrieval stage VERIFIED.")


## 4. LLM + RAG combined — grounded answers, hard-required

System prompt rules the model on:
- opening with the consent disclosure,
- pitching the programme,
- answering fee/date/certification questions **only** from retrieved context (refusing to
  guess if the context doesn't cover it),
- probing closing interest.

Opt-out / "do not call" requests are handled **outside** the LLM entirely (see the
`is_opt_out_request` check below) — that's a compliance action, not a conversational one,
and must not depend on the LLM choosing to comply.

In [ ]:
SYSTEM_PROMPT_TEMPLATE = f'''You are {AGENT_DISPLAY_NAME}, an AI voice agent for {COMPANY_NAME}, calling to \
pitch the "{PROGRAMME_NAME}" training programme and gauge the caller's interest.

Rules you must always follow:
1. The call opened with a clear disclosure that this is an AI-conducted, recorded call \
(that disclosure has already been played to the caller — do not repeat it unless asked).
2. You will be given "RETRIEVED CONTEXT" before each caller message. When the caller asks \
about the fee, dates/schedule, duration, curriculum, or certification, you MUST base your \
answer ONLY on that retrieved context. If the retrieved context does not contain the answer, \
say you'll confirm the detail and follow up — NEVER invent or guess a fee, date, or \
certification detail.
3. For general questions unrelated to the programme, answer briefly and helpfully, then \
steer the conversation back to the programme.
4. Keep responses short and conversational (2-4 sentences), as this is a spoken phone call, \
not a written chat.
5. Naturally probe the caller's level of interest (are they interested, do they want more \
info sent, would they like to enroll) as the conversation progresses.
6. Be respectful of objections. If the caller is hesitant, answer their concern once; do not \
pressure them repeatedly.
'''

OPT_OUT_PHRASES = [
    "remove my number", "take me off", "stop calling", "don't call me", "do not call me",
    "do not call", "unsubscribe", "opt out", "opt-out", "stop contacting", "remove me from",
    "don't contact me", "do not contact me",
]

def is_opt_out_request(text: str) -> bool:
    lowered = text.lower()
    return any(phrase in lowered for phrase in OPT_OUT_PHRASES)

SUPPRESSION_LIST = []  # in a real system this would write to a do-not-call datastore

def generate_response(user_text: str, chat_history=None):
    '''
    Returns a dict: {reply, suppressed, context_used, retrieval_hits, elapsed}.
    chat_history: list of {"role": "user"/"assistant", "content": ...} from earlier turns.
    '''
    chat_history = chat_history or []
    t0 = time.time()

    if is_opt_out_request(user_text):
        SUPPRESSION_LIST.append(user_text)
        reply = (
            "Understood — I've flagged this number to be removed from our calling list "
            "immediately, and you will not receive further calls about this programme. "
            "Sorry for the interruption, and thank you for your time."
        )
        return {
            "reply": reply,
            "suppressed": True,
            "context_used": None,
            "retrieval_hits": [],
            "elapsed": time.time() - t0,
        }

    context_text, hits = retrieve_context(user_text, k=2)
    messages = [{"role": "system", "content": SYSTEM_PROMPT_TEMPLATE}]
    messages += chat_history
    messages.append({
        "role": "user",
        "content": f"RETRIEVED CONTEXT:\n{context_text}\n\nCALLER SAID: {user_text}",
    })

    reply, _ = llm_raw_generate(messages, max_new_tokens=180, temperature=0.4)
    return {
        "reply": reply,
        "suppressed": False,
        "context_used": context_text,
        "retrieval_hits": hits,
        "elapsed": time.time() - t0,
    }

print("generate_response() ready.")


In [ ]:
# --- Verify grounded factual answers before wiring up TTS ---
for q in ["What's the fee?", "When does it start?", "Do I get a certificate?"]:
    result = generate_response(q)
    print(f"Q: {q}\n  suppressed={result['suppressed']}  ({result['elapsed']:.2f}s)\n  A: {result['reply']}\n")

opt_out_result = generate_response("Please remove my number, I'm not interested")
print(f"Opt-out check -> suppressed={opt_out_result['suppressed']}")
assert opt_out_result["suppressed"] is True, "Opt-out request did not trigger suppression!"
print("LLM+RAG grounded-response stage VERIFIED.")


## 5. Text-to-Speech — Piper

Using real, current Piper voice names from the `rhasspy/piper-voices` Hugging Face repo
(verified, not assumed) — `en_US-lessac-medium` and `en_GB-alan-medium`. There is no
`en_IN` voice in Piper's catalog, so we don't use one.

In [ ]:
import wave
from piper import PiperVoice

PIPER_VOICE_NAME = "en_US-lessac-medium"   # alt: "en_GB-alan-medium"
PIPER_VOICE_DIR = "/content/piper_voices"
os.makedirs(PIPER_VOICE_DIR, exist_ok=True)

_voice_lang_dir = {"en_US-lessac-medium": "en/en_US/lessac/medium",
                    "en_GB-alan-medium": "en/en_GB/alan/medium"}[PIPER_VOICE_NAME]
_base_url = f"https://huggingface.co/rhasspy/piper-voices/resolve/main/{_voice_lang_dir}"

onnx_path = os.path.join(PIPER_VOICE_DIR, f"{PIPER_VOICE_NAME}.onnx")
config_path = os.path.join(PIPER_VOICE_DIR, f"{PIPER_VOICE_NAME}.onnx.json")

for fname, path in [(f"{PIPER_VOICE_NAME}.onnx", onnx_path), (f"{PIPER_VOICE_NAME}.onnx.json", config_path)]:
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{_base_url}/{fname}", path)

# --- Verify the downloaded voice files are real, non-empty files before loading them ---
onnx_size = os.path.getsize(onnx_path)
config_size = os.path.getsize(config_path)
print(f"{onnx_path}: {onnx_size} bytes")
print(f"{config_path}: {config_size} bytes")
assert onnx_size > 10_000_000, f"Voice model file looks too small ({onnx_size} bytes) — download likely failed."
assert config_size > 100, f"Voice config file looks too small ({config_size} bytes) — download likely failed."

# CPU inference: the `piper-tts` pip package installs plain `onnxruntime` (not
# `onnxruntime-gpu`), so use_cuda=True here would fail with no CUDA execution
# provider available. TTS for short spoken responses is fast enough on CPU;
# the GPU is reserved for the LLM and Whisper above.
piper_voice = PiperVoice.load(onnx_path, config_path=config_path, use_cuda=False)
print("Piper voice loaded and verified:", PIPER_VOICE_NAME)


In [ ]:
def synthesize_speech(text: str, out_path: str = "/content/tts_output.wav"):
    '''Returns (out_path, elapsed_seconds).'''
    t0 = time.time()
    with wave.open(out_path, "wb") as wav_file:
        piper_voice.synthesize_wav(text, wav_file)
    elapsed = time.time() - t0
    size = os.path.getsize(out_path)
    assert size > 1000, f"TTS output looks empty ({size} bytes) — investigate before continuing."
    return out_path, elapsed

# --- Verify TTS on both the fixed opening script and a sample LLM-style response ---
from IPython.display import Audio, display

for label, sample_text in [("opening consent script", CONSENT_DISCLOSURE),
                            ("sample LLM response", "The programme fee is fully explained on request.")]:
    path, elapsed = synthesize_speech(sample_text, out_path=f"/content/tts_test_{label[:4]}.wav")
    print(f"[{label}] synthesized in {elapsed:.2f}s -> {path} ({os.path.getsize(path)} bytes)")
    display(Audio(path))

print("TTS stage VERIFIED.")


## 6. Chain — STT → RAG → LLM → TTS in one call

Takes either typed text or a recorded/uploaded audio file, runs the full chain, plays back
the spoken result, and prints per-stage latency every time.

In [ ]:
def run_pipeline(text: str = None, audio_path: str = None, chat_history=None, play_audio=True, verbose=True):
    '''
    Exactly one of `text` or `audio_path` should be given.
    Returns a dict with the transcript (if any), the reply, suppression flag, and timings.
    '''
    assert (text is None) != (audio_path is None), "Pass exactly one of text= or audio_path="
    timings = {"stt": 0.0, "llm": 0.0, "tts": 0.0}
    t_start = time.time()

    if audio_path is not None:
        text, stt_elapsed = transcribe_audio(audio_path)
        timings["stt"] = stt_elapsed
        if verbose:
            print(f"[STT {stt_elapsed:.2f}s] \"{text}\"")

    llm_result = generate_response(text, chat_history=chat_history)
    timings["llm"] = llm_result["elapsed"]
    if verbose:
        tag = "SUPPRESSED (opt-out)" if llm_result["suppressed"] else "reply"
        print(f"[LLM {timings['llm']:.2f}s] ({tag}) \"{llm_result['reply']}\"")

    tts_path, tts_elapsed = synthesize_speech(llm_result["reply"], out_path="/content/pipeline_output.wav")
    timings["tts"] = tts_elapsed
    if verbose:
        print(f"[TTS {tts_elapsed:.2f}s] -> {tts_path}")
        if play_audio:
            display(Audio(tts_path))

    timings["total"] = time.time() - t_start
    if verbose:
        print(f"[Total {timings['total']:.2f}s]  (stt={timings['stt']:.2f}s  llm={timings['llm']:.2f}s  tts={timings['tts']:.2f}s)")

    return {
        "user_text": text,
        "reply": llm_result["reply"],
        "suppressed": llm_result["suppressed"],
        "retrieval_hits": llm_result["retrieval_hits"],
        "timings": timings,
    }

# Quick smoke test of the full chain with typed text:
_ = run_pipeline(text="What's the fee for the programme?")


## 7. Test harness — sanity-check the full chain in one run

Runs a fixed set of caller questions/objections through `run_pipeline` and prints each
result, so after any change you can re-run this one cell and eyeball whether behavior
regressed — including confirming the "remove my number" case produces a suppression
signal rather than a normal conversational reply.

In [ ]:
TEST_CASES = [
    ("fee question",        "How much does this course cost?"),
    ("dates question",      "When does the batch start and how long does it run?"),
    ("certification question", "Will I get a certificate after finishing?"),
    ("opt-out / DNC",       "I'm not interested, please remove my number from your list."),
    ("curveball",           "Random question — what's the capital of Australia?"),
]

def run_test_harness():
    print(f"Running {len(TEST_CASES)} test cases through the full chain...\n" + "=" * 70)
    results = []
    for label, caller_text in TEST_CASES:
        print(f"\n--- [{label}] caller says: \"{caller_text}\" ---")
        result = run_pipeline(text=caller_text, play_audio=False, verbose=False)
        print(f"  reply: {result['reply']}")
        print(f"  suppressed: {result['suppressed']}")
        print(f"  timings: stt={result['timings']['stt']:.2f}s llm={result['timings']['llm']:.2f}s "
              f"tts={result['timings']['tts']:.2f}s total={result['timings']['total']:.2f}s")
        results.append((label, result))

    # Explicit pass/fail check on the one hard behavioral requirement: opt-out must suppress.
    opt_out_result = dict(results)["opt-out / DNC"]
    status = "PASS" if opt_out_result["suppressed"] else "FAIL"
    print("\n" + "=" * 70)
    print(f"[{status}] opt-out request produced suppression signal: {opt_out_result['suppressed']}")
    assert opt_out_result["suppressed"], "Opt-out case did not suppress — this must be fixed before going further."
    return results

_ = run_test_harness()


## Next steps (not in this notebook)

This proof-of-concept validates the local pipeline end to end. The next phase — out of
scope here — wires this into a self-hosted PBX/SIP trunk for real calls, adds streaming
STT/TTS for lower latency, and adds the offline post-call analysis pass that scores/
classifies each transcript (Interested / Likely / Hold / Not Interested / Invalid contact).